# ðŸ§  LifeOS: Training an LLM to Survive a Chaotic Week

**GRPO Training with TRL + Unsloth on the LifeOS OpenEnv Environment**

This notebook trains an LLM to handle cascading personal life conflicts using:
- **GRPO** (Group Relative Policy Optimization) â€” no value model needed
- **Unsloth** â€” 4-bit LoRA for memory-efficient training
- **4 independent reward functions** â€” task completion, social coherence, energy sustainability, format compliance

Theme: **Personalized Tasks (#3.2)** | Stack: TRL, Unsloth, OpenEnv

## 1. Setup â€” Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece
!pip install matplotlib numpy pydantic fastapi requests

In [ ]:
# Clone the LifeOS repo (if running on Colab)
import os
if not os.path.exists('LifeOS'):
    !git clone https://github.com/itzzSPcoder/LifeOS.git

import sys
sys.path.insert(0, '/content/LifeOS')
os.chdir('/content/LifeOS')

## 2. Load the LifeOS Environment

In [ ]:
from lifeos.envs.student_week_openenv import StudentWeekEnv, Action, VALID_ACTION_TYPES
from lifeos.training.train_grpo import build_prompt, parse_llm_output, heuristic_action

# Quick test
env = StudentWeekEnv(max_steps=30)
obs = env.reset()
print(f"Environment ready! Energy={obs.energy}, Stress={obs.stress}")
print(f"Tasks: {len(obs.tasks)}, Messages: {len(obs.inbox)}")
print(f"Action space: {list(VALID_ACTION_TYPES)}")

## 3. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: {MODEL_NAME} (4-bit LoRA, r=16)")

## 4. Run Heuristic Baseline

In [ ]:
NUM_EPISODES = 50

baseline_rewards = []
for ep in range(NUM_EPISODES):
    env = StudentWeekEnv(max_steps=30)
    obs = env.reset()
    done = False
    while not done:
        action = heuristic_action(obs.model_dump())
        obs, reward, done, info = env.step(action)
    baseline_rewards.append(env.state.total_reward)

baseline_mean = sum(baseline_rewards) / len(baseline_rewards)
print(f"Heuristic baseline mean reward: {baseline_mean:.3f}")

## 5. Build Training Dataset

In [ ]:
from datasets import Dataset

# Generate diverse prompts from environment resets
prompts = []
for _ in range(NUM_EPISODES):
    env = StudentWeekEnv(max_steps=30)
    obs = env.reset()
    prompts.append(build_prompt(obs.model_dump()))

dataset = Dataset.from_dict({"prompt": prompts})
print(f"Dataset: {len(dataset)} prompts")
print(f"Sample prompt (first 300 chars):\n{prompts[0][:300]}...")

## 6. Define Reward Function (4 Independent Signals)

In [ ]:
def lifeos_reward_fn(completions: list[str], prompts: list[str], **kwargs) -> list[float]:
    """
    Reward function for GRPO. Runs each completion through the environment
    and returns the composite reward from all 4 independent signals.
    """
    rewards = []
    for completion in completions:
        env = StudentWeekEnv(max_steps=30, chaos_probability=0.0)
        env.reset()
        action = parse_llm_output(completion)
        _obs, reward, _done, info = env.step(action)
        rewards.append(reward)
    return rewards

# Quick test
test_rewards = lifeos_reward_fn(
    ['{"action_type": "rest"}', '{"action_type": "INVALID"}'],
    ['test', 'test']
)
print(f"Test rewards: {test_rewards}")
print("(First should be positive, second should be negative due to format penalty)")

## 7. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

ADAPTER_OUTPUT = "outputs/lifeos-grpo-adapter"

grpo_config = GRPOConfig(
    output_dir=ADAPTER_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    max_completion_length=256,
    num_generations=4,
    logging_steps=1,
    save_steps=NUM_EPISODES,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    reward_funcs=lifeos_reward_fn,
)

print("Starting GRPO training...")
trainer.train()
print("Training complete!")

## 8. Save LoRA Adapter (NOT merged 4-bit)

In [ ]:
import os
os.makedirs(ADAPTER_OUTPUT, exist_ok=True)
model.save_pretrained(ADAPTER_OUTPUT)
tokenizer.save_pretrained(ADAPTER_OUTPUT)
print(f"LoRA adapter saved to {ADAPTER_OUTPUT}/")
print("NOTE: Saved as adapter only (not merged) to preserve quality.")

## 9. Post-Training Evaluation

In [ ]:
FastLanguageModel.for_inference(model)

trained_rewards = []
component_rewards = {"task_completion": [], "social_coherence": [],
                     "energy_sustainability": [], "format_compliance": []}

for ep in range(min(NUM_EPISODES, 30)):
    env = StudentWeekEnv(max_steps=30)
    obs = env.reset()
    done = False
    while not done:
        prompt = build_prompt(obs.model_dump())
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=256,
                                  temperature=0.7, do_sample=True)
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        action = parse_llm_output(text)
        obs, reward, done, info = env.step(action)
    state = env.state
    trained_rewards.append(state.total_reward)
    for key in component_rewards:
        component_rewards[key].append(state.reward_components.get(key, 0.0))
    if ep % 10 == 0:
        print(f"Eval {ep+1} | reward={state.total_reward:.3f} | "
              f"tasks={state.tasks_completed} | msgs={state.messages_answered}")

trained_mean = sum(trained_rewards) / len(trained_rewards)
print(f"\nTrained agent mean reward: {trained_mean:.3f}")
print(f"Baseline mean reward:     {baseline_mean:.3f}")
print(f"Improvement:              {trained_mean - baseline_mean:+.3f}")

## 10. Reward Curves

In [ ]:
import matplotlib.pyplot as plt
import os

os.makedirs("outputs", exist_ok=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top: Composite reward
ax1.plot(trained_rewards, label="GRPO Agent", color="#6366f1", linewidth=2)
ax1.axhline(y=baseline_mean, color="#ef4444", linestyle="--", linewidth=1.5,
             label=f"Heuristic Baseline ({baseline_mean:.2f})")
ax1.set_ylabel("Composite Reward")
ax1.set_title("LifeOS Training â€” Composite Reward per Episode")
ax1.legend()
ax1.grid(alpha=0.2)

# Bottom: Individual reward functions
colors = {"task_completion": "#10b981", "social_coherence": "#3b82f6",
          "energy_sustainability": "#f59e0b", "format_compliance": "#8b5cf6"}
for key, values in component_rewards.items():
    ax2.plot(values, label=key.replace("_", " ").title(),
             color=colors.get(key, "#999"), linewidth=1.5, alpha=0.8)
ax2.set_xlabel("Training Episode")
ax2.set_ylabel("Component Reward")
ax2.set_title("Individual Reward Function Curves")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig("outputs/reward_curves.png", dpi=150)
plt.show()
print("Saved to outputs/reward_curves.png")

## 11. Before/After Qualitative Example

In [ ]:
# Same scenario, no chaos â€” compare heuristic vs trained
env = StudentWeekEnv(max_steps=30, chaos_probability=0.0)
obs = env.reset()
prompt = build_prompt(obs.model_dump())

# Heuristic
h_action = heuristic_action(obs.model_dump())
print("=" * 60)
print("BEFORE TRAINING (Heuristic Agent)")
print("=" * 60)
print(f"Action: {h_action.action_type}({h_action.target_id})")
print(f"Tone: {h_action.tone}")
print(f"Reason: {h_action.reason}")

# Trained
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
t_action = parse_llm_output(text)

print("\n" + "=" * 60)
print("AFTER TRAINING (GRPO Agent)")
print("=" * 60)
print(f"Action: {t_action.action_type}({t_action.target_id})")
print(f"Tone: {t_action.tone}")
print(f"Reason: {t_action.reason}")
print(f"\nRaw model output (first 500 chars):")
print(text[:500])

## Summary

| Component | Details |
|---|---|
| **Environment** | LifeOS StudentWeekEnv (OpenEnv-compliant) |
| **Algorithm** | GRPO (Group Relative Policy Optimization) |
| **Model** | Mistral-7B-Instruct-v0.3 (4-bit LoRA via Unsloth) |
| **Reward Signals** | Task Completion, Social Coherence, Energy Sustainability, Format Compliance |
| **Anti-Hack** | Step timeout, loop detection, protected state check |
| **Output** | LoRA adapter saved to `outputs/lifeos-grpo-adapter/` |